In [1]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_expenses.csv to cleaned_expenses.csv


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ExpenseAnomalyDetection").getOrCreate()

df = spark.read.csv("cleaned_expenses.csv", header=True, inferSchema=True)
df = df.withColumn("month", F.date_format("month", "yyyy-MM"))

monthly = df.groupBy("user_id", "month").agg(F.sum("amount").alias("monthly_total"))
print("Monthly spend per user:")
monthly.orderBy("user_id", "month").show()

user_window = Window.partitionBy("user_id")
spikes = (
    monthly
    .withColumn("avg_monthly_total", F.avg("monthly_total").over(user_window))
    .filter(F.col("monthly_total") > 1.5 * F.col("avg_monthly_total"))
)
print("Users with unusual monthly spikes:")
spikes.show()

large_expenses = (
    df
    .withColumn("avg_amount", F.avg("amount").over(user_window))
    .filter(F.col("amount") > 2 * F.col("avg_amount"))
    .select("user_id", "date", "category", "amount", "description")
)
print("Users with large one-time expenses:")
large_expenses.show()

spark.stop()

Monthly spend per user:
+-------+-------+------------------+
|user_id|  month|     monthly_total|
+-------+-------+------------------+
|      1|2025-01|29138.940000000006|
|      1|2025-02|          20146.24|
|      1|2025-03|49750.020000000004|
|      1|2025-04| 43676.49999999999|
|      1|2025-05|16111.010000000004|
|      1|2025-06|           43047.4|
|      1|2025-07|          20583.83|
|      1|2025-08| 97724.88999999998|
|      1|2025-09|          44356.43|
|      1|2025-10|40674.049999999996|
|      1|2025-11|37468.509999999995|
|      1|2025-12|          22404.48|
|      1|2026-01|19713.350000000006|
|      1|2026-02|26580.049999999996|
|      1|2026-03|28714.650000000005|
|      1|2026-04|23649.640000000007|
|      1|2026-05| 91088.11000000002|
|      1|2026-06|23121.760000000002|
|      1|2026-07| 64431.97000000001|
|      1|2026-08|          41743.03|
+-------+-------+------------------+
only showing top 20 rows
Users with unusual monthly spikes:
+-------+-------+-----------